# RF center angular correlogram

Read saved 2D RF centers for one session/probe. E and I mean excitatory and inhibitory RF regions.
Within each neuron, calculate target azimuth minus reference azimuth: E–E uses both directions and excludes self-pairs; E→I uses I minus E. Distinct centers at the same azimuth remain separate.

Pool raw pair counts across all saved units, with no HD/ADN filter, then apply circular Gaussian smoothing for the line plot. Neurons with more centers contribute more pairs. This is a descriptive angular distribution; elevation is not used. Detection results are only read.


In [ ]:
%matplotlib inline

from pathlib import Path
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from RF.profiles import load_rf_centers
from RF.rflocate import rf_result_path
from Utils.plotting import apply_light_plot_style
from Utils.statistic_utils import smooth_tuning_curve

apply_light_plot_style()


In [ ]:
base_dir = Path("/mnt/senzailab/Kai/#Recording/m22")
date = 261008
sessionID = 2
probe_name = "A"
bins = 30  # Equal angular bins over 360 degrees.
smooth_sigma = 1.0  # Gaussian SD in angular bins (12 degrees at 30 bins).

session_dir = base_dir / str(date) / f"{date}_{sessionID}"
rf_dir = session_dir / "data/rfmapping/good/-100_400_1ms" / f"Probe{probe_name}"
source = rf_dir / f"regular_unitsSpikeCounts_{date}_{sessionID}.rfmap"


In [ ]:
centers = {}
for label, rf_type in (("E", "excitatory"), ("I", "inhibitory")):
    table = load_rf_centers(
        source, probe=probe_name, detected_rf_path=rf_result_path(source, rf_type=rf_type),
    ).reset_index()
    table["center_id"] = table.groupby(["probe", "unit_id"]).cumcount()
    centers[label] = table

print(source)
print(f"Saved centers: E = {len(centers['E'])}, I = {len(centers['I'])}")


In [ ]:
pairs = {}
for label, target in (("E–E", "E"), ("E→I", "I")):
    table = centers["E"].merge(
        centers[target], on=["probe", "unit_id"], suffixes=("_ref", "_target"),
    )
    if target == "E":
        table = table.loc[table.center_id_ref != table.center_id_target].copy()
    table["delta_deg"] = (table.rf_x_deg_target - table.rf_x_deg_ref + 180) % 360 - 180
    pairs[label] = table


In [ ]:
bin_width = 360 / bins
edges = np.linspace(0, 360, bins + 1) - bin_width / 2  # Center a bin at zero.
theta = np.deg2rad((edges[:-1] + edges[1:]) / 2)
fig, axes = plt.subplots(1, 2, figsize=(10, 5), subplot_kw={"projection": "polar"}, layout="constrained")

for ax, (label, table), color in zip(axes, pairs.items(), ("#248f55", "#8055a5")):
    angles = (table.delta_deg.to_numpy() + bin_width / 2) % 360 - bin_width / 2
    counts, _ = np.histogram(angles, bins=edges)
    curve = smooth_tuning_curve(pd.Series(counts), sigma=smooth_sigma).to_numpy()
    ax.plot(np.r_[theta, theta[0] + 2 * np.pi], np.r_[curve, curve[0]], color=color, linewidth=2)
    ax.set_ylim(bottom=0)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_thetagrids([0, 60, 120, 180, 240, 300], ["0°", "+60°", "+120°", "180°", "−120°", "−60°"])
    n_units = len(table[["probe", "unit_id"]].drop_duplicates())
    ax.set_title(f"{label} | {n_units} neurons | {len(table)} ordered pairs", pad=20)

fig.suptitle(f"{base_dir.name} / {date}_{sessionID} / Probe{probe_name}\nSmoothed pair count · Gaussian σ = {smooth_sigma:g} bins ({smooth_sigma * bin_width:g}°)")
plt.show()
